In [2]:
from langchain_postgres import PGVectorStore, PGEngine
from langchain_openai import OpenAIEmbeddings
import os

embeddings = OpenAIEmbeddings(
    model=os.environ['EMBEDDING_MODEL_NAME'],
    base_url=os.environ['BASE_URL'],
    api_key=os.environ['OPENAI_API_KEY'],
    check_embedding_ctx_length=False
)

engine = PGEngine.from_connection_string(url=os.environ['POSTGRES_URL'])

vector_store = PGVectorStore.create_sync(engine=engine, embedding_service=embeddings, table_name='langchain_test')
vector_store.similarity_search(query="客服", k=2)

[Document(id='68bdc2f4-2e9b-4d3b-aa86-2b09ec49156a', metadata={'source': '../documents/data.txt'}, page_content='案例：电商平台使用虚拟导购助手，通过WebRTC与用户实时互动，推荐商品并解答问题。'),
 Document(id='e7bd0178-4710-4968-935a-a32e71638c9d', metadata={'source': '../documents/data.txt'}, page_content='三、虚拟助手与实时互动\n应用场景：在线客服、虚拟导购、智能教育助手。\n技术实现：WebRTC提供实时音视频交互能力，AI通过语音识别和自然语言处理理解用户需求，并提供智能回复或引导。')]

In [ ]:
vector_store.as_retriever(search_kwargs=dict(k=2)).invoke(input="导购")

In [ ]:
from langchain.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI
import os
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

store_retriever = vector_store.as_retriever(search_kwargs=dict(k=1))

llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=store_retriever,
    llm=llm,
)

retriever_from_llm.invoke('导购')

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)
# 创建一个从 Document 中提取核心内容的 compressor
compressor = LLMChainExtractor.from_llm(llm)
# 创建一个会自动对上下文进行压缩的 Retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vector_store.as_retriever(search_kwargs=dict(k=2))
)
# vector_store.as_retriever(search_kwargs=dict(k=2)).invoke("安防")

compression_retriever.invoke("安防")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

SYSTEM_TEMPLATE='''
你是一个熟知内部知识库的机器人，你在回答时会引用知识库，并擅长通过自己的总结归纳，组织语言给出答案。
并且回答时仅根据知识库，尽可能回答用户问题，如果知识库中没有相关内容，你可以回答“原文中没有相关内容”，不要回答知识库以外的内容。

以下是知识库中跟用户回答相关的内容：
{context}

现在，你需要基于知识库，回答以下问题：
{question}
'''

prompt = ChatPromptTemplate.from_template(template=SYSTEM_TEMPLATE)

convert_docs_to_string = lambda docs: "".join(
    [f"{doc.page_content}\n" for doc in docs]
)

retriever_chain = compression_retriever | RunnableLambda(convert_docs_to_string)

def input_to_context(input):
    return {"question": input, "context": retriever_chain.invoke(input=input)}

rag_chain = RunnableSequence(
    first=RunnableLambda(input_to_context),
    middle=[
        prompt,
        llm,
    ],
    last=StrOutputParser(),
)

rst = rag_chain.invoke(input='客服')
print(rst)